In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

import sys

sys.path.append("..")

from src.backtester import calculate_return, backtester

In [2]:
df = pd.read_csv("../data/spy_daily.csv")
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Capital Gains
0,2000-01-03 00:00:00-05:00,148.25000,148.25000,143.875000,145.4375,91.132774,8164300,0.0,0.0,0.0
1,2000-01-04 00:00:00-05:00,143.53125,144.06250,139.640625,139.7500,87.568886,8089800,0.0,0.0,0.0
2,2000-01-05 00:00:00-05:00,139.93750,141.53125,137.250000,140.0000,87.725510,12177900,0.0,0.0,0.0
3,2000-01-06 00:00:00-05:00,139.62500,141.50000,137.750000,137.7500,86.315620,6227200,0.0,0.0,0.0
4,2000-01-07 00:00:00-05:00,140.31250,145.75000,140.062500,145.7500,91.328552,8066500,0.0,0.0,0.0


In [3]:
prices = df["Adj Close"]
print(len(prices))

6713


#### Defining a Momentum Generating Signal

In [4]:
def generate_momentum_signal(prices, lookback):

    momentum_signal = [np.nan] * lookback

    for i in range(lookback, len(prices)):
        roc = (prices.iloc[i] - prices.iloc[i - lookback]) / prices.iloc[i - lookback]

        if roc > 0:
            signal = 1
        else:
            signal = 0

        momentum_signal.append(signal)

    return momentum_signal

In [29]:
df["Momentum Signal"] = generate_momentum_signal(prices, 252)

print((df["Momentum Signal"] == 1).sum())
print((df["Momentum Signal"] == 0).sum())

momentum_signal = df["Momentum Signal"]
print(len(momentum_signal))

5009
1452
6713


In [23]:
returns = calculate_return(prices)
print(len(returns))

6713


#### Backtest

In [30]:
momentum_df, momentum_results = backtester(returns, momentum_signal, 0.001)
pd.Series(momentum_results)

total_return          5.400420
cagr                  0.075102
annual_volatility     0.128810
annual_sharpe         0.627024
max_drawdown         -0.315812
calmar                0.237805
time_invested         0.775232
trades               89.000000
dtype: float64

#### Buy & Hold Strategy Comparison

In [14]:
def generate_buy_hold_signal(prices, lookback):

    buy_hold_signal = [np.nan] * lookback

    for i in range(lookback, len(prices)):
        buy_hold_signal.append(1)

    return buy_hold_signal

In [15]:
df["Buy Hold Signal"] = generate_buy_hold_signal(prices, 252)
buy_hold_signal = df["Buy Hold Signal"]
print(len(buy_hold_signal))

6713


In [17]:
buy_hold_df, buy_hold_results = backtester(returns, buy_hold_signal, 0.001)
pd.Series(buy_hold_results)

total_return         8.361472
cagr                 0.091168
annual_volatility    0.190530
annual_sharpe        0.553301
max_drawdown        -0.551895
calmar               0.165190
time_invested        1.000000
trades               1.000000
dtype: float64